# Day 2 Reference Solution: Patient, prescription & clinician (CLN)

**JUDGES ONLY. Never share with teams.**

This is the full registry the CLN starter is asking teams to build toward: **four Kind 1 patterns** (Patient, Prescription, Clinician, Lab) and **four Kind 2 candidates deliberately rejected** (Ward, Wing, Dose, Shift), each with a reason.

It runs the public pack *and* the hidden pack. Because the starter's `grounded_answer` is fixed (record IDs in the context, the checked verdict beside each pair, `temperature=0`), the answer follows the graph verdict whenever retrieval returns the two records, so the **registry is what is being judged**.

The offline checker (`day2_offline_check.py --dataset CLN`) reproduces the structural verdicts without Neo4j or an LLM.

## 1. Setup

In [ ]:
%pip install -q sentence-transformers hnswlib openai python-dotenv neo4j

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import hnswlib
import os

def get_secret(name):
    """Reads a secret from Colab's Secrets manager (padlock icon in the left
    sidebar) if running on Colab; falls back to a local .env file otherwise."""
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.getenv(name)

model = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model ready")

### Your LLM provider

Bring your own key (any provider) and add it to Colab's Secrets manager (padlock icon) as `LLM_API_KEY`. Every LLM call in this notebook goes through the single function `call_llm` below.

Run the cell: it should print a one-word reply. If it errors, fix this cell before going on.

In [ ]:
# ---- YOUR LLM PROVIDER (your own key, any provider) ----
# Add your key to Colab Secrets as LLM_API_KEY. The whole notebook talks to the model through
# ONE function, call_llm(prompt) -> str, so this is the only cell you change for a new provider.
# The default below works for OpenAI and for any provider with an OpenAI-compatible endpoint:
# set LLM_MODEL (and LLM_BASE_URL if it is not OpenAI) from your provider's documentation.
# A provider with its own SDK: keep the name and signature of call_llm, rewrite the body.
from openai import OpenAI

LLM_MODEL = "gpt-4o-mini"
LLM_BASE_URL = None  # None = OpenAI. Otherwise your provider's OpenAI-compatible URL

client = OpenAI(api_key=get_secret("LLM_API_KEY"), base_url=LLM_BASE_URL)

def call_llm(prompt):
    """Sends one prompt, returns the model's reply as plain text.
    Keep temperature=0: otherwise the same question can give a different answer on each run."""
    response = client.chat.completions.create(
        model=LLM_MODEL, temperature=0,
        messages=[{"role": "user", "content": prompt}])
    return response.choices[0].message.content

print("LLM check:", call_llm("Reply with the single word: ready"))

### Connect to Neo4j Aura

Create a free [Neo4j Aura](https://neo4j.com/cloud/aura-free/) instance if you haven't, then add `NEO4J_URI`, `NEO4J_USER` and `NEO4J_PASSWORD` to Colab's Secrets manager (padlock icon). If you hit `ServiceUnavailable`, the instance auto-paused: re-run this cell.

**Cannot use Neo4j?** Skip this cell. Replace the graph-building cell (section 7) and `graph_check` (section 8) with lookup-table versions that keep the same return shape. The Student Setup Guide shows the table structure.

In [ ]:
from neo4j import GraphDatabase

uri = get_secret("NEO4J_URI")
user = get_secret("NEO4J_USER")
password = get_secret("NEO4J_PASSWORD")

driver = GraphDatabase.driver(uri, auth=(user, password))

def run_write(query, **params):
    with driver.session() as session:
        session.run(query, **params)

def run_read(query, **params):
    with driver.session() as session:
        result = session.run(query, **params)
        return [record.data() for record in result]

print(run_read("RETURN 'Connection successful' AS message")[0]["message"])

## 2. The dataset: campus operations & student services

179 records in 14 categories. Four pairs of categories deliberately share vocabulary (money, rooms, passes/permits, events). Most records stand alone; some name a person, a club or a ticket.

In [ ]:
DATASET_ID = "CLN"
DATASET_NAME = "Patient, prescription & clinician"

rec_categories = {
    'Appointments': [
        'A follow-up appointment was booked for next week',
        "The clinic rescheduled a morning slot after a doctor's leave was approved",
        'An appointment reminder was sent by text message',
        'A walk-in visit was converted into a scheduled appointment',
        'Online booking was paused during a system upgrade',
        'A cancelled appointment slot was offered to the waiting list',
        'An interpreter was arranged for a scheduled visit',
        'A telehealth consultation link was sent to the visitor',
        'Appointment no-shows were reviewed at the weekly meeting',
        'Patient P2063 booked a follow-up appointment after the first visit',
        'Clinician DR12 approved a leave and the slot was rescheduled',
        'Wing B was reserved for the orthopaedic outpatient sessions',
    ],
    'Registration': [
        'A new visitor completed the registration form at the front desk',
        'An identity document was scanned and attached to the record',
        'A duplicate registration record was merged by the records team',
        "The front desk updated a visitor's contact details",
        'Self-service kiosks were added to the registration area',
        'A registration queue formed during the morning rush',
        'Consent forms were collected at registration',
        "A visitor's insurance details were captured at the desk",
        'The registration team corrected a misspelt name',
        'Patient P2044 registered at the front desk and completed the consent forms',
    ],
    'Consultations': [
        "The doctor reviewed the visitor's history before the consultation",
        'A specialist referral was written after the examination',
        'A second opinion was requested by the family',
        'The consultation notes were typed into the clinical system',
        'A dietitian advised on a meal plan during the visit',
        'Follow-up advice was shared in a printed summary',
        'A paediatric consultation ran over the scheduled time',
        'The doctor explained the test results in plain language',
        'A short consultation was held by phone',
        'Prescription RX114 was written after the examination',
        'Clinician DR07 held the consultation and wrote a specialist referral',
        'Prescription RX101 was revised by the doctor after the interaction was flagged',
        'Prescription RX122 was written for a 250 mg dose of an antibiotic',
        'A 500 mg antibiotic course was prescribed for a throat infection',
    ],
    'Lab_Tests': [
        'Blood samples were collected and labelled at the phlebotomy desk',
        'The lab analyser was recalibrated after a quality check',
        'A urine sample was rejected for being insufficient',
        "Test results were released to the doctor's inbox",
        'A critical value was phoned through to the ward',
        'The lab ran a batch of cholesterol panels',
        'A sample was recollected after a labelling error',
        'External reference lab results were received by courier',
        'Reagent stock was replenished after a shortage',
        'Sample LAB-3305 was collected and labelled at the phlebotomy desk',
        "Sample LAB-3305 results were released to the doctor's inbox",
        "Patient P2058's blood test was repeated after a labelling error",
    ],
    'Imaging': [
        'A chest X-ray was reported by the radiologist',
        'The MRI scanner was serviced by the vendor',
        'An ultrasound appointment was moved to the afternoon',
        'A CT scan was repeated because of patient movement',
        'Imaging reports were uploaded to the clinical system',
        'A radiographer completed a safety check on the equipment',
        'A contrast allergy was noted before a scan',
        'The imaging queue was reorganised for urgent cases',
        'An old film archive was digitised',
        'Clinician DR07 requested an urgent CT scan',
        'Patient P2058 had a chest X-ray reported by the radiologist',
    ],
    'Pharmacy': [
        'The pharmacy dispensed a repeat prescription',
        'A drug interaction was flagged by the pharmacist',
        'Pharmacy stock of antibiotics was replenished',
        'A generic substitute was offered for an unavailable brand',
        'Expired medicines were removed from the shelves',
        'Labels with dosage instructions were printed for a dispensing',
        'The pharmacy confirmed a prescription with the clinic',
        'Cold chain storage for vaccines was checked',
        'A pharmacist counselled a visitor on how to take a medicine',
        'Prescription RX114 was dispensed by the pharmacy',
        'Prescription RX101 was flagged by the pharmacist for a drug interaction',
        'Prescription RX122 was refilled at the pharmacy',
        'A 500 mg paracetamol tablet was dispensed to a walk-in visitor',
        'A batch of 250 mg tablets was recalled by the manufacturer',
    ],
    'Billing_Insurance': [
        "A consultation fee was posted to the visitor's account",
        'An insurance pre-authorisation was requested',
        'A claim was rejected for a missing form',
        'A co-payment was collected at the front desk',
        'The billing team corrected a coding error on a claim',
        'An insurer queried the length of a hospital stay',
        'A bill estimate was shared before a scheduled procedure',
        'A claim reimbursement was received from the insurer',
        'A payment plan was arranged for an outstanding bill',
        "Patient P2044's insurance claim was submitted after the consultation",
        'Patient P2077 was given a bill estimate before the procedure',
        'Billing queries peaked after the evening clinic',
    ],
    'Wards_Admissions': [
        'A bed was allocated to a person admitted through emergency',
        'Admission paperwork was completed at the bedside',
        'A transfer to the intensive care unit was arranged',
        'Bed occupancy was reviewed during the morning rounds',
        'A planned admission was postponed by the surgeon',
        'A relative was given visiting hour instructions',
        'A shortage of beds triggered a diversion request',
        'Ward rounds were completed before noon',
        'A person was moved to a single room for isolation',
        'Patient P2031 was admitted through emergency and allocated a bed',
        'Patient P2058 was moved out of Ward 3 to a single room',
        'A bed shortage was reported in Ward 6',
    ],
    'Discharge': [
        'A discharge summary was prepared for the family doctor',
        'Discharge was delayed while awaiting a pharmacy supply',
        'Home care instructions were explained to the family',
        'A follow-up visit was arranged at discharge',
        'Transport was booked for a person going home',
        'The discharge lounge was opened to free up beds',
        'A discharge against advice was documented',
        'The billing clearance was completed before discharge',
        'A nurse reviewed the discharge checklist with the family',
        'Patient P2063 was discharged with home care instructions',
    ],
    'Nursing': [
        'Nursing handover was completed at the change of shift',
        'Vital signs were recorded every four hours',
        'A fall-risk assessment was repeated for an elderly person',
        'A nurse administered scheduled medication',
        'Wound dressings were changed according to the care plan',
        'A nurse escalated a rise in temperature',
        'Nursing staffing was adjusted for the weekend',
        'Pressure sore prevention rounds were documented',
        'A nurse trained a family member to give injections',
        'Patient P2031 had a fall-risk assessment repeated',
        'Patient P2077 complained about noise in Ward 3 at night',
        'Extra nurses were rostered for the evening clinic',
    ],
    'Facilities': [
        'The waiting area chairs were replaced',
        'A leaking tap in the restroom was repaired',
        'The generator was tested before the monsoon',
        'Signage was updated for the new entrance',
        'The lift was out of service for repairs',
        'Cleaning contractors changed their schedule',
        'Biomedical engineering calibrated the monitors',
        'Parking for visitors was extended',
        'An air conditioner was serviced in the waiting hall',
        'The generator in Wing B failed during the monsoon storm',
    ],
    'Infection_Control': [
        'Hand hygiene audits were carried out on all floors',
        'A cluster of infections prompted extra cleaning',
        'Isolation signs were placed outside a room',
        'Surgical instruments were sterilised in the central unit',
        'Staff were reminded to wear masks in clinical areas',
        'Waste segregation was inspected by the infection control nurse',
        'A vaccine drive was run for staff',
        'A sterilisation cycle was repeated after a failed indicator',
        'Environmental swabs were sent to the lab',
        'Ward 6 was closed for deep cleaning after the infection cluster',
        'Clinician DR19 led the infection control audit',
        'Swab batch LAB-2201 was sent to the lab for culture',
    ],
}

rec_texts = []
rec_labels = []
for category, sentences in rec_categories.items():
    for s in sentences:
        rec_texts.append(s)
        rec_labels.append(category)

print(f"Dataset {DATASET_ID}: {len(rec_texts)} records, {len(rec_categories)} categories")

## 3. Retrieval (given: this part works)

In [ ]:
rec_embeddings = model.encode(rec_texts, show_progress_bar=True)
rec_embeddings = np.ascontiguousarray(rec_embeddings.astype('float32'))

dim = rec_embeddings.shape[1]
rec_index = hnswlib.Index(space='cosine', dim=dim)
rec_index.init_index(max_elements=len(rec_texts), ef_construction=200, M=16)
rec_index.add_items(rec_embeddings, np.arange(len(rec_texts)))
rec_index.set_ef(50)
print(f"Index built: {len(rec_texts)} records")


def retrieve(question, k=3):
    q_emb = model.encode([question])[0].astype('float32').reshape(1, -1)
    labels, distances = rec_index.knn_query(q_emb, k=k)
    return [(int(idx), rec_texts[idx], rec_labels[idx]) for idx in labels[0]]

## 4. Baseline RAG (given: no grounding)

The plain Week 2 pipeline. It retrieves similar records and trusts the LLM to reason about whether they are connected.

In [ ]:
# The plain Week 2 prompt -- no grounding. temperature=0 so runs are repeatable.
BASELINE_PROMPT = """Answer the question using the context below.

Context:
{context}

Question: {question}

Answer:"""

def baseline_rag_answer(question, k=3):
    results = retrieve(question, k=k)
    context = "\n".join(f"- {text}" for _, text, _ in results)
    prompt = BASELINE_PROMPT.format(context=context, question=question)
    return {"question": question, "retrieved": results, "answer": call_llm(prompt)}

## 5. The pattern registry (the answer key)

In [ ]:
import re

PATTERNS = [
    {
        "name": 'Patient',
        "regex": r"\bPatient\s+(P\d{4})\b",   # group 1 = the identifier
        "kind": 1,
        "example": 'Patient P2044',
        "why": 'A patient ID names one person: the registration and the insurance claim for P2044 are about the same patient.',
    },
    {
        "name": 'Prescription',
        "regex": r"\bPrescription\s+(RX\d{3})\b",   # group 1 = the identifier
        "kind": 1,
        "example": 'Prescription RX114',
        "why": 'A prescription number names one prescription from writing to dispensing.',
    },
    {
        "name": 'Clinician',
        "regex": r"\bClinician\s+(DR\d{2})\b",   # group 1 = the identifier
        "kind": 1,
        "example": 'Clinician DR07',
        "why": 'A clinician ID names one doctor; the consultation and the CT request for DR07 are about the same clinician.',
    },
    {
        "name": 'Lab',
        "regex": r"\b(LAB-\d{3,4})\b",   # group 1 = the identifier
        "kind": 1,
        "example": 'LAB-3305',
        "why": 'A lab sample number names one sample from collection to result.',
    },
]

REJECTED_CANDIDATES = [
    {"candidate": 'Ward numbers (Ward 3, Ward 6)', "kind": 2, "why_rejected": 'A ward is a place that many patients pass through: P2058 moving out of Ward 3 and P2077 complaining about noise in Ward 3 are unrelated events.'},
    {"candidate": 'Wing letters (Wing B)', "kind": 2, "why_rejected": 'A wing is a part of the building shared by many events: outpatient sessions and a generator failure in Wing B are separate.'},
    {"candidate": 'Dose strengths (250 mg, 500 mg)', "kind": 2, "why_rejected": "Equal strengths appear in unrelated records: RX122's 250 mg prescription and an unrelated 250 mg batch recall."},
    {"candidate": 'Clinic session (evening clinic)', "kind": 2, "why_rejected": 'A session name is a recurring time slot shared by many unrelated records.'},
]

def validate_registry():
    """Checks that every pattern is complete and compiles. Run this before building the graph."""
    problems = []
    for p in PATTERNS:
        label = p.get("name", "?")
        for key in ("name", "regex", "kind", "example", "why"):
            if p.get(key) in (None, ""):
                problems.append(f"{label}: missing '{key}'")
        if p.get("name") and not re.fullmatch(r"[A-Za-z][A-Za-z0-9_]*", p["name"]):
            problems.append(f"{label}: name must be letters/digits/underscore only")
        try:
            rx = re.compile(p.get("regex", ""))
            if rx.groups < 1:
                problems.append(f"{label}: regex needs ONE capture group around the identifier")
        except re.error as e:
            problems.append(f"{label}: regex does not compile ({e})")
        if p.get("kind") not in (1, 2):
            problems.append(f"{label}: kind must be 1 or 2")
    return problems

def show_register():
    print(f"{'Pattern':<10} {'Kind':<5} {'Example':<20} Why")
    for p in PATTERNS:
        print(f"{p['name']:<10} {p['kind']:<5} {p['example']:<20} {p['why']}")
    if REJECTED_CANDIDATES:
        print("\nConsidered but rejected:")
        for r in REJECTED_CANDIDATES:
            print(f"  - {r['candidate']} (Kind {r['kind']}): {r['why_rejected']}")

problems = validate_registry()
print("Registry problems:", problems if problems else "none")
show_register()

## 6. Build the graph

In [ ]:
def extract_entities(text):
    """Applies every registered pattern. Returns [(pattern_name, identifier), ...]."""
    found = []
    for p in PATTERNS:
        for m in re.finditer(p["regex"], text):
            ent = (p["name"], m.group(1))
            if ent not in found:
                found.append(ent)
    return found

assert not validate_registry(), "Fix the registry problems above first"

run_write("MATCH (n) DETACH DELETE n")

# One :Event node per record
event_rows = [{"id": f"R{i}", "text": t, "category": c}
              for i, (t, c) in enumerate(zip(rec_texts, rec_labels))]
run_write("""
    UNWIND $rows AS row
    CREATE (e:Event {id: row.id, text: row.text, category: row.category})
""", rows=event_rows)

# One entity node per distinct identifier; records naming it are linked through it
by_label = {}
for i, text in enumerate(rec_texts):
    for name, ident in extract_entities(text):
        by_label.setdefault(name, []).append({"eid": f"R{i}", "xid": ident})

for name, rows in by_label.items():       # name was validated: letters/digits/underscore only
    run_write(f"""
        UNWIND $rows AS r
        MATCH (e:Event {{id: r.eid}})
        MERGE (x:{name} {{id: r.xid}})
        MERGE (e)-[:MENTIONS]->(x)
    """, rows=rows)

print("Graph built from your registry:")
for name, rows in by_label.items():
    ids = {}
    for r in rows:
        ids.setdefault(r["xid"], 0)
        ids[r["xid"]] += 1
    shared = sum(1 for c in ids.values() if c > 1)
    print(f"  {name:<10} {len(rows):>3} mentions | {len(ids):>3} distinct identifiers | {shared} shared by 2+ records (= links)")

## 7. Grounded answer

In [ ]:
def graph_check(id_a, id_b):
    """Asks Neo4j whether a path connects two records, and through which entities."""
    rows = run_read("""
        MATCH (a:Event {id: $a}), (b:Event {id: $b})
        OPTIONAL MATCH p = shortestPath((a)-[*..6]-(b))
        RETURN p IS NULL AS disconnected,
               [n IN nodes(p) WHERE NOT n:Event | labels(n)[0] + ' ' + n.id] AS via
    """, a=id_a, b=id_b)
    return rows[0] if rows else None


GROUNDED_PROMPT = """You answer questions about outpatient clinic and hospital operations (appointments, labs, pharmacy, wards, billing) records. Use ONLY the records and graph facts below.

Rules:
1. The question's wording is a claim to check, NOT evidence. Do not accept a link just because the question assumes one.
2. Doses, ward numbers, wing letters, clinic sessions, dates and similar shared values are NOT links.
3. The graph facts are ground truth about links. If the two records the question is about are NOT CONNECTED,
   say plainly that no recorded link exists between them. If they are CONNECTED, say so and name what they share.
4. If the records do not answer the question, say so. Cite record IDs like [R12].

Records:
{context}

Graph facts (checked in the database):
{graph_facts}

Question: {question}

Answer:"""


def grounded_answer(question, k=3):
    results = retrieve(question, k=k)
    ids = [f"R{idx}" for idx, _, _ in results]

    fact_lines, verified_connect, verified_disconnect = [], False, False
    for i in range(len(ids)):
        for j in range(i + 1, len(ids)):
            chk = graph_check(ids[i], ids[j])
            if chk is None:
                continue
            if chk["disconnected"]:
                verified_disconnect = True
                fact_lines.append(f"- [{ids[i]}] and [{ids[j]}]: NOT CONNECTED -- no shared named entity links them.")
            else:
                verified_connect = True
                fact_lines.append(f"- [{ids[i]}] and [{ids[j]}]: CONNECTED -- via {', '.join(chk['via'] or [])}.")

    context = "\n".join(f"- [R{idx}] {text}" for idx, text, _ in results)
    prompt = GROUNDED_PROMPT.format(context=context,
                                    graph_facts="\n".join(fact_lines) or "(no pairs to check)",
                                    question=question)
    return {"question": question, "retrieved": results, "graph_facts": fact_lines,
            "verified_connect": verified_connect, "verified_disconnect": verified_disconnect,
            "answer": call_llm(prompt)}

## 8. Question packs

In [ ]:
PUBLIC_ADVERSARIAL = [{'id': 'P1',
  'question': 'Was the walk-in who was converted to an appointment the same visitor who completed '
              'registration at the front desk?',
  'trap_records': ['A walk-in visit was converted into a scheduled appointment',
                   'A new visitor completed the registration form at the front desk'],
  'correct_answer_must_state': 'No stated connection: neither record names a patient ID, so they '
                               'cannot be shown to be the same visitor.',
  'expect_connected': False,
  'kind': 'Vocabulary overlap (visit)'},
 {'id': 'P2',
  'question': 'Were the cholesterol panels ordered because of the chest X-ray?',
  'trap_records': ['The lab ran a batch of cholesterol panels',
                   'A chest X-ray was reported by the radiologist'],
  'correct_answer_must_state': 'No stated connection: a lab batch and an imaging report name no '
                               'shared patient or sample.',
  'expect_connected': False,
  'kind': 'Vocabulary overlap (tests)'},
 {'id': 'P3',
  'question': 'Was the insurance claim rejected because a generic substitute was offered?',
  'trap_records': ['A generic substitute was offered for an unavailable brand',
                   'A claim was rejected for a missing form'],
  'correct_answer_must_state': 'No stated connection: the claim was rejected for a missing form, '
                               'and no patient links it to the substitute.',
  'expect_connected': False,
  'kind': 'Vocabulary overlap (claims)'},
 {'id': 'P4',
  'question': 'Did the delayed discharge cause the shortage of beds that triggered the diversion '
              'request?',
  'trap_records': ['Discharge was delayed while awaiting a pharmacy supply',
                   'A shortage of beds triggered a diversion request'],
  'correct_answer_must_state': 'No stated connection: neither record names a patient or ward.',
  'expect_connected': False,
  'kind': 'Vocabulary overlap (beds)'},
 {'id': 'P5',
  'question': 'Was the bed shortage in Ward 6 caused by the deep cleaning closure of Ward 6?',
  'trap_records': ['Ward 6 was closed for deep cleaning after the infection cluster',
                   'A bed shortage was reported in Ward 6'],
  'correct_answer_must_state': 'No stated connection: both mention Ward 6, but a ward is a place '
                               'used by many events, not an event identifier.',
  'expect_connected': False,
  'kind': 'Kind 2 trap: ward'},
 {'id': 'P6',
  'question': 'Did the orthopaedic outpatient sessions in Wing B cause the generator failure?',
  'trap_records': ['Wing B was reserved for the orthopaedic outpatient sessions',
                   'The generator in Wing B failed during the monsoon storm'],
  'correct_answer_must_state': 'No stated connection: sharing a wing name is not sharing an event.',
  'expect_connected': False,
  'kind': 'Kind 2 trap: wing'},
 {'id': 'P7',
  'question': 'Was the 500 mg paracetamol dispensed against the 500 mg antibiotic prescription?',
  'trap_records': ['A 500 mg paracetamol tablet was dispensed to a walk-in visitor',
                   'A 500 mg antibiotic course was prescribed for a throat infection'],
  'correct_answer_must_state': 'No stated connection: the strengths match by coincidence, and the '
                               'drugs and records differ.',
  'expect_connected': False,
  'kind': 'Kind 2 trap: dose'},
 {'id': 'P8',
  'question': 'Was a second opinion requested because the lift was out of service?',
  'trap_records': ['A second opinion was requested by the family',
                   'The lift was out of service for repairs'],
  'correct_answer_must_state': 'No stated connection between a second opinion and a lift outage; '
                               'any causal story is invented.',
  'expect_connected': False,
  'kind': 'Cross-category causal'}]

PUBLIC_POSITIVE = [{'id': 'PP1',
  'question': 'Did Patient P2044 register and consent at the front desk and later have an '
              'insurance claim submitted?',
  'shared_identifier': 'Patient P2044',
  'shared_records': ['Patient P2044 registered at the front desk and completed the consent forms',
                     "Patient P2044's insurance claim was submitted after the consultation"],
  'correct_answer_must_state': 'Yes -- both records name Patient P2044.',
  'expect_connected': True},
 {'id': 'PP2',
  'question': 'Did Patient P2063 book a follow-up and also get discharged with home care '
              'instructions?',
  'shared_identifier': 'Patient P2063',
  'shared_records': ['Patient P2063 booked a follow-up appointment after the first visit',
                     'Patient P2063 was discharged with home care instructions'],
  'correct_answer_must_state': 'Yes -- both records name Patient P2063.',
  'expect_connected': True},
 {'id': 'PP3',
  'question': 'Was Prescription RX114 written after the examination and then dispensed by the '
              'pharmacy?',
  'shared_identifier': 'Prescription RX114',
  'shared_records': ['Prescription RX114 was written after the examination',
                     'Prescription RX114 was dispensed by the pharmacy'],
  'correct_answer_must_state': 'Yes -- both records name Prescription RX114.',
  'expect_connected': True}]

print(f"Public pack: {len(PUBLIC_ADVERSARIAL)} adversarial + {len(PUBLIC_POSITIVE)} positive-control questions")

In [ ]:
HIDDEN_ADVERSARIAL = [{'id': 'H1',
  'question': "Did the insurer's query about the hospital stay come from the test results released "
              "to the doctor's inbox?",
  'trap_records': ["Test results were released to the doctor's inbox",
                   'An insurer queried the length of a hospital stay'],
  'correct_answer_must_state': 'No stated connection: no patient or sample is named in both '
                               'records.',
  'expect_connected': False,
  'kind': None},
 {'id': 'H2',
  'question': 'Were appointment no-shows reviewed because the vaccine cold chain storage was '
              'checked?',
  'trap_records': ['Appointment no-shows were reviewed at the weekly meeting',
                   'Cold chain storage for vaccines was checked'],
  'correct_answer_must_state': 'No stated connection between an appointment review and a '
                               'cold-chain check.',
  'expect_connected': False,
  'kind': None},
 {'id': 'H3',
  'question': 'Was the contrast allergy noted before the scan the same issue as the drug '
              'interaction the pharmacist flagged?',
  'trap_records': ['A contrast allergy was noted before a scan',
                   'A drug interaction was flagged by the pharmacist'],
  'correct_answer_must_state': 'No stated connection: neither record names a patient or '
                               'prescription, so they cannot be shown to be the same issue.',
  'expect_connected': False,
  'kind': None},
 {'id': 'H4',
  'question': 'Did the hand hygiene audits lead to the co-payment collected at the front desk?',
  'trap_records': ['Hand hygiene audits were carried out on all floors',
                   'A co-payment was collected at the front desk'],
  'correct_answer_must_state': 'No stated connection between an infection-control audit and a '
                               'billing record.',
  'expect_connected': False,
  'kind': None},
 {'id': 'H5',
  'question': 'Was the telehealth link sent because the imaging reports were uploaded?',
  'trap_records': ['A telehealth consultation link was sent to the visitor',
                   'Imaging reports were uploaded to the clinical system'],
  'correct_answer_must_state': 'No stated connection: no shared patient is named in both records.',
  'expect_connected': False,
  'kind': None},
 {'id': 'H6',
  'question': 'Did extending visitor parking relate to the doctor explaining the test results?',
  'trap_records': ['Parking for visitors was extended',
                   'The doctor explained the test results in plain language'],
  'correct_answer_must_state': 'No connection: a facilities change and a consultation record do '
                               'not share vocabulary; any causal story is fabricated.',
  'expect_connected': False,
  'kind': None},
 {'id': 'H7',
  'question': 'Was the noise in Ward 3 that Patient P2077 complained about the reason Patient '
              'P2058 was moved out of Ward 3?',
  'trap_records': ['Patient P2058 was moved out of Ward 3 to a single room',
                   'Patient P2077 complained about noise in Ward 3 at night'],
  'correct_answer_must_state': 'No -- two different patients (P2058, P2077) mention Ward 3, but a '
                               'ward is a place used by many events, not an event identifier. '
                               'Linking by ward would wrongly connect them.',
  'expect_connected': False,
  'kind2_trap': 'Ward number (Ward 3)'},
 {'id': 'H8',
  'question': 'Was the recalled 250 mg batch the one used for Prescription RX122?',
  'trap_records': ['Prescription RX122 was written for a 250 mg dose of an antibiotic',
                   'A batch of 250 mg tablets was recalled by the manufacturer'],
  'correct_answer_must_state': 'Prescription RX122 is for a 250 mg dose, and a 250 mg batch recall '
                               'exists, but the recall names no prescription. The matching '
                               'strength is incidental, not an event identifier.',
  'expect_connected': False,
  'kind2_trap': 'Dose (250 mg)'},
 {'id': 'H9',
  'question': 'Were billing queries peaking because extra nurses were rostered for the evening '
              'clinic?',
  'trap_records': ['Billing queries peaked after the evening clinic',
                   'Extra nurses were rostered for the evening clinic'],
  'correct_answer_must_state': 'No stated connection: both mention the evening clinic, a recurring '
                               'session shared by many unrelated records.',
  'expect_connected': False,
  'kind2_trap': 'Session (evening clinic)'}]

HIDDEN_POSITIVE = [{'id': 'HP1',
  'question': 'Was sample LAB-3305 collected and its results later released to the doctor?',
  'shared_identifier': 'LAB-3305',
  'shared_records': ['Sample LAB-3305 was collected and labelled at the phlebotomy desk',
                     "Sample LAB-3305 results were released to the doctor's inbox"],
  'correct_answer_must_state': 'Yes -- both records carry sample LAB-3305.',
  'expect_connected': True},
 {'id': 'HP2',
  'question': 'Was Patient P2031 admitted through emergency and also given a repeated fall-risk '
              'assessment?',
  'shared_identifier': 'Patient P2031',
  'shared_records': ['Patient P2031 was admitted through emergency and allocated a bed',
                     'Patient P2031 had a fall-risk assessment repeated'],
  'correct_answer_must_state': 'Yes -- both records name Patient P2031.',
  'expect_connected': True},
 {'id': 'HP3',
  'question': 'Did Clinician DR07 hold the consultation and also request the urgent CT scan?',
  'shared_identifier': 'Clinician DR07',
  'shared_records': ['Clinician DR07 held the consultation and wrote a specialist referral',
                     'Clinician DR07 requested an urgent CT scan'],
  'correct_answer_must_state': 'Yes -- both records name Clinician DR07.',
  'expect_connected': True},
 {'id': 'HP4',
  'question': 'Was Prescription RX101 flagged for a drug interaction and then revised by the '
              'doctor?',
  'shared_identifier': 'Prescription RX101',
  'shared_records': ['Prescription RX101 was flagged by the pharmacist for a drug interaction',
                     'Prescription RX101 was revised by the doctor after the interaction was '
                     'flagged'],
  'correct_answer_must_state': 'Yes -- both records name Prescription RX101.',
  'expect_connected': True}]

print(f"Hidden pack: {len(HIDDEN_ADVERSARIAL)} adversarial + {len(HIDDEN_POSITIVE)} positive-control questions")

In [ ]:
def structural_check(test):
    """No LLM needed: does the graph connect the two records this question is about?
    Returns (passed, connected, via)."""
    recs = test.get("trap_records") or test.get("shared_records")
    ia, ib = [f"R{rec_texts.index(t)}" for t in recs]
    chk = graph_check(ia, ib)
    connected = not chk["disconnected"]
    return connected == test["expect_connected"], connected, chk["via"]


def run_pack(pack, show_answers=True):
    """Runs baseline and grounded side by side on every question in a pack."""
    rows = []
    for t in pack:
        ok, connected, via = structural_check(t)
        print(f"[{t['id']}] {t['question']}")
        if show_answers:
            print("   Baseline :", baseline_rag_answer(t["question"])["answer"][:350].replace("\n", " "))
            print("   Grounded :", grounded_answer(t["question"])["answer"][:350].replace("\n", " "))
        print(f"   Graph    : {'CONNECTED' if connected else 'NOT connected'}"
              f"{' via ' + ', '.join(via) if via else ''} | expected "
              f"{'connected' if t['expect_connected'] else 'NOT connected'} -> {'PASS' if ok else 'FAIL'}")
        print("   Must say :", t["correct_answer_must_state"][:200])
        print()
        rows.append({"id": t["id"], "graph_pass": ok, "connected": connected})
    print(f"Graph verdict correct on {sum(r['graph_pass'] for r in rows)}/{len(rows)}")
    return rows

### 8a. Public pack

In [ ]:
pub_adv = run_pack(PUBLIC_ADVERSARIAL)
pub_pos = run_pack(PUBLIC_POSITIVE)

### 8b. HIDDEN pack (judges)

Expect 9/9 adversarial NOT connected and 4/4 positive CONNECTED at the graph level. Read the LLM answers as well: they should say "no recorded link" for adversarial questions, and affirm and name the shared entity for positives. Where baseline and grounded disagree is the story to tell.

In [ ]:
hid_adv = run_pack(HIDDEN_ADVERSARIAL)
hid_pos = run_pack(HIDDEN_POSITIVE)

## 9. Teaching points

1. **Kind 1 vs Kind 2.** Kind 2 traps: Ward 3 (H7), the 250 mg dose (H8), the evening clinic (H9); public pack has Ward 6, Wing B and 500 mg. A team that adds those patterns fails exactly those questions.
2. **Coverage.** Hidden positives need Lab (HP1), Clinician (HP3) and Prescription (HP4) on top of Patient (HP2). A team that only adds the first obvious pattern fails the others.
3. **A baseline that fails only sometimes.** With a strict prompt the baseline can pass many of these questions; with the plain Week 2 prompt it is more likely to fabricate. Compare their answers on P5-P7 and H7-H9, and report what you actually see.
4. **Retrieval is separate.** If retrieval does not return both records, the graph check never sees them. Report that as a retrieval miss, not a grounding failure.